# C4 — Prompt Engineering pentru adnotare

Corpusul: `data/cleaned/corpus_youtube_sample.json` — text brut, fara etichete.  
Promptul: `prompts/annotation_prompt.md` — fisier comun, completat de echipa.  
Output: `data/annotated/` — adnotarile produse azi, input pentru tipologii in C5.


## 0. Setup

In [25]:
import os, json, re, random
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
# caută .env urcând din folderul curent
ROOT = Path.cwd()
while not (ROOT / ".env").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
load_dotenv(ROOT / ".env")

# DeepSeek
deepseek_client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com"
)
DEEPSEEK_MODEL = "deepseek-chat"
# Gemini prin OpenAI-compatible API
gemini_client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

GEMINI_MODEL = "gemini-2.5-flash-lite"
# alegem modelul pentru demo

USE_GEMINI = True
client_now = gemini_client if USE_GEMINI else deepseek_client
model_now = GEMINI_MODEL if USE_GEMINI else DEEPSEEK_MODEL
print("Root project:", ROOT)
print("DeepSeek key:", os.getenv("DEEPSEEK_API_KEY") is not None)
print("Gemini key:", os.getenv("GEMINI_API_KEY") is not None)
print("Model folosit:", model_now)
print("OK")

Root project: c:\Users\rotar\Desktop\echochamber-project-team-5\echochamber-project-team-5
DeepSeek key: True
Gemini key: True
Model folosit: gemini-2.5-flash-lite
OK


## 1. Corpusul — text brut

In [26]:
import pandas as pd
import random

corpus = pd.read_json("../../data/cleaned/corpus_youtube_sample.jsonl", lines=True)

print(len(corpus), "comentarii")
print("Câmpuri:", list(corpus.columns))

for _, c in corpus.sample(3).iterrows():
    print(f"[{c['source_channel'][:30]}] {c['text'][:80]}")

420 comentarii
Câmpuri: ['id', 'source_channel', 'video_title', 'text']
[RecorderRomania] Numai minciuni prezentati ! minciunile prezentate de Bruxeles si Ucraina le baga
[DianaSosoacaOfficial] Nu mor că slujesc lui Satan și o duc foarte bine s-au adunat cu schismaticii și 
[turcescu111] Pai suntem in faliment ! Au început cu plandemia vieții LOR si de atunci încoace


In [27]:
corpus.head(2)

,id,source_channel,video_title,text
0,yt_Vekhmz5OPCc_UgzpXOYhxlT3Nqo6h7J4AaABAg,NicusorDanRO,🟢 LIVE Declarații de presă susținute la Palatu...,jigodia aia de Georgescu nu lua intrebari inca...
1,yt_olpOFshMJD0_UgznnPuVIJ6HcddfPbN4AaABAg,georgesimionoficial,#democratie #georgesimion #unitate #prosperita...,Mă voi realizați că votul s-a încheiat și Nicu...


## 2. Promptul meu — citit din fisier

Promptul complet e in `prompts/annotation_prompt.md`.  
Deschideti fisierul si cititi-l — contine rolul, principiul metodologic, vocabularul, definitiile, registrele, regulile si 6 exemple complete.  
Sectiunile `TODO` le completati voi mai tarziu.


In [28]:
with open('../../prompts/annotation_prompt (1).md', encoding='utf-8') as f:
    SYSTEM_PROMPT = f.read()

print(f'Prompt incarcat: {len(SYSTEM_PROMPT):,} caractere')
print(f'Sectiuni principale:')
for line in SYSTEM_PROMPT.split('\n'):
    if line.startswith('## '):
        print(f'  {line}')

Prompt incarcat: 11,139 caractere
Sectiuni principale:
  ## SYSTEM
  ## PRINCIPIU METODOLOGIC
  ## VOCABULAR TARGET
  ## CÂMPURI DE BAZĂ
  ## CELE 5 AXE DISCURSIVE
  ## REGULI DE CODARE
  ## EXEMPLE DE BAZĂ
  ## FORMAT OUTPUT


## 3. Tehnici de prompt engineering

Folosim acelasi comentariu de test pe toate tehnicile ca sa vedem diferenta.


In [29]:
TEST = corpus.sample(1).iloc[0]
print(f"Canal:  {TEST['source_channel']}")
print(f"Text:   {TEST['text']}")

Canal:  RecorderRomania
Text:   Acum 4 ani si in cazul Alexandrei Macesanu politia a fost la fel de indolenta, ulterior alte zeci de cazuri la fel, acum la fel. Oare chiar nimeni nu vrea sa schimbe nimic? Oare de cate ori trebuie sa se repete?


### Tehnica 1 — Zero-shot
Nicio instructiune. Modelul ghiceste ce vrei — format impredictibil.


In [30]:

SYSTEM = "Esti analist de discurs politic. Raspunde scurt si clar."
USER = TEST["text"]
response = client_now.chat.completions.create(
    model=model_now,
    temperature=0,
    messages=[
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": USER}
    ]
)
print(response.choices[0].message.content)

Situația semnalată ridică întrebări legitime despre eficacitatea și promptitudinea instituțiilor statului în gestionarea unor cazuri grave. Repetarea unor astfel de evenimente, în ciuda tragediilor, sugerează o problemă sistemică ce necesită o analiză aprofundată a procedurilor, resurselor și culturii organizaționale din cadrul instituțiilor implicate.


### Tehnica 2 — Role + format explicit
Rol clar + template gol. Modelul stie ce campuri vrei, dar nu stie logica.


In [31]:
SYSTEM_SIMPLE = """
Esti un coder de discurs politic romanesc.
Adnoteaza comentariul strict pe baza textului.
Returneaza doar JSON valid, fara explicatii.

Format:
{
  "target": "",
  "stance": "pro | anti | neutru | ambiguu | none",
  "inst_neg": 0,
  "repr_personalist": 0,
  "confidence": 0.0
}
"""
USER = TEST["text"]

response = client_now.chat.completions.create(
    model=model_now,
    temperature=0,
    max_tokens=300,
    messages=[
        {"role": "system", "content": SYSTEM_SIMPLE},
        {"role": "user", "content": USER}
    ]
)
print(response.choices[0].message.content)

```json
{
  "target": "politia",
  "stance": "anti",
  "inst_neg": 1,
  "repr_personalist": 0,
  "confidence": 0.9
}
```


### Tehnica 3 — Promptul complet
Acum nu mai dăm doar un rol și un template gol.
Folosim promptul complet din `annotation_prompt.md`, care conține:
- vocabular controlat pentru `target`;
- definiții pentru `stance` și `tone`;
- axe discursive numerice;
- reguli de codare;
- exemple few-shot.
Modelul nu decide direct bula discursivă. El adnotează comentariul pe axe empirice. Tipologia/bulele vor fi construite ulterior printr-un script.

In [32]:
def llm(system, user, max_tokens=700):
    response = client_now.chat.completions.create(
        model=model_now,
        temperature=0,
        max_tokens=max_tokens,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user}
        ]
    )
    return response.choices[0].message.content

In [33]:
import json
# 1. Comentariul + context minim

USER = f"""
CANAL:
{TEST.get("source_channel", "")}
TITLU VIDEO:
{TEST.get("video_title", "")}
COMENTARIU:
<<< {TEST["text"]} >>>
"""

# 2. Apel la model
raw = llm(SYSTEM_PROMPT, USER)
# 3. Curățare răspuns JSON
json_text = (
    raw.replace("```json", "")
       .replace("```", "")
       .strip()
)

# 4. Transformare în dicționar Python
r = json.loads(json_text)
# 5. Afișare
print("Comentariu:")
print(TEST["text"])
print("\nAdnotare:")
for key in ["target", "stance", "tone", "institutional", "legitimare", "epistemic", "geopolitic", "mobilizare", "confidence"]:
    print(f"{key}: {r[key]}")
print("\nJustificare:")
print(r["justification"])

Comentariu:
Acum 4 ani si in cazul Alexandrei Macesanu politia a fost la fel de indolenta, ulterior alte zeci de cazuri la fel, acum la fel. Oare chiar nimeni nu vrea sa schimbe nimic? Oare de cate ori trebuie sa se repete?

Adnotare:
target: justitie
stance: anti
tone: acuzator
institutional: -1
legitimare: 0
epistemic: -1
geopolitic: 0
mobilizare: 0
confidence: 0.9

Justificare:
Comentariul acuză poliția de indolenta repetată în mai multe cazuri, sugerând o problemă sistemică.


### Tehnica 4 — Structured Output
Schema trimisa ca parametru API — modelul **nu poate** returna altceva.  
Nu mai avem nevoie de `extract_json()`. Output garantat valid.


In [34]:
SCHEMA = {
    "type": "json_schema",
    "json_schema": {
        "name": "annotation",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "target": {
                    "type": "string",
                    "enum": [
                        "georgescu", "simion", "aur", "sosoaca",
                        "psd", "pnl", "usr", "nicusor_dan", "bolojan", "other_mainstream_actor",
                        "guvern", "presedintie", "parlament", "ccr", "alegeri", "justitie", "other_state_institution",
                        "ue", "nato", "bruxelles", "other_external_actor",
                        "recorder", "g4media", "digi24", "presa_mainstream", "presa_investigativa", "other_media",
                        "none"
                    ]
                },
                "stance": {
                    "type": "string",
                    "enum": ["pro", "anti", "neutru", "ambiguu", "none"]
                },
                "tone": {
                    "type": "string",
                    "enum": ["acuzator", "ironic", "mobilizator", "defensiv", "afectiv", "neutru"]
                },
                "institutional": {
                    "type": "integer",
                    "enum": [-2, -1, 0, 1, 2]
                },
                "legitimare": {
                    "type": "integer",
                    "enum": [-2, -1, 0, 1, 2]
                },
                "epistemic": {
                    "type": "integer",
                    "enum": [-2, -1, 0, 1, 2]
                },
                "geopolitic": {
                    "type": "integer",
                    "enum": [-2, -1, 0, 1, 2]
                },
                "mobilizare": {
                    "type": "integer",
                    "enum": [0, 1, 2]
                },
                "justification": {
                    "type": "string"
                },
                "confidence": {
                    "type": "number",
                    "minimum": 0,
                    "maximum": 1
                }
            },
            "required": [
                "target",
                "stance",
                "tone",
                "institutional",
                "legitimare",
                "epistemic",
                "geopolitic",
                "mobilizare",
                "justification",
                "confidence"
            ],
            "additionalProperties": False
        }
    }
}

In [35]:
def llm_structured(system, user):
    response = client_now.chat.completions.create(
        model=model_now,
        temperature=0,
        max_tokens=700,
        response_format=SCHEMA,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user}
        ]
    )
    return json.loads(response.choices[0].message.content)

In [36]:
USER = f"""
CANAL:
{TEST.get("source_channel", "")}
TITLU VIDEO:
{TEST.get("video_title", "")}
COMENTARIU:
<<< {TEST["text"]} >>>
"""
r = llm_structured(SYSTEM_PROMPT, USER)
print("Comentariu:")
print(TEST["text"])
print()
print("Adnotare structurată:")
for key in [
    "target",
    "stance",
    "tone",
    "institutional",
    "legitimare",
    "epistemic",
    "geopolitic",
    "mobilizare",
    "confidence",
]:
    print(f"{key}: {r[key]}")
print()
print("Justificare:")
print(r["justification"])
print()
print(f"Câmpuri garantate: {len(r)} — fără extract_json()")

Comentariu:
Acum 4 ani si in cazul Alexandrei Macesanu politia a fost la fel de indolenta, ulterior alte zeci de cazuri la fel, acum la fel. Oare chiar nimeni nu vrea sa schimbe nimic? Oare de cate ori trebuie sa se repete?

Adnotare structurată:
target: other_state_institution
stance: anti
tone: acuzator
institutional: -1
legitimare: 0
epistemic: 0
geopolitic: 0
mobilizare: 0
confidence: 0.9

Justificare:
Comentariul acuză poliția de 

Câmpuri garantate: 10 — fără extract_json()


# II. Typology building

In [37]:
annotated_path = Path("../../data/annotated/corpus_axis_annotated.jsonl")
typed_path = Path("../data/typed/corpus_axis_typed.jsonl")
typed_path.parent.mkdir(parents=True, exist_ok=True)

df = pd.read_json(annotated_path, lines=True)

print(df.shape)
df.head()

(420, 16)


,id,source_channel,channel_family,video_title,text,target,stance,tone,institutional,legitimare,epistemic,geopolitic,mobilizare,justification,confidence,fallback
0,yt_Vekhmz5OPCc_UgzpXOYhxlT3Nqo6h7J4AaABAg,NicusorDanRO,,🟢 LIVE Declarații de presă susținute la Palatu...,jigodia aia de Georgescu nu lua intrebari inca...,georgescu,anti,acuzator,0,0,-1,0,0,Comentariul acuză pe Georgescu de intenții rel...,0.90,False
1,yt_olpOFshMJD0_UgznnPuVIJ6HcddfPbN4AaABAg,georgesimionoficial,,#democratie #georgesimion #unitate #prosperita...,Mă voi realizați că votul s-a încheiat și Nicu...,nicusor_dan,pro,neutru,1,1,0,0,0,Comentariul afirmă că Nicușor Dan a fost ales ...,0.90,False
2,yt_RU8h59PnVDM_Ugxw6Vquj0Br3S0VMGV4AaABAg,@CălinGeorgescu-CanalulOficial,,Călin Georgescu - AJUNGE! A trecut un an...( ...,Lepra aia care ia interval este non stop lângă...,georgescu,anti,acuzator,0,0,0,0,0,Comentariul folosește metafora 'lepra' pentru ...,0.85,False
3,yt_sXqW2Cnw0DU_UgyFd08cuCXGNjBPclx4AaABAg,euronewsro,,Vicepreședinte ANCMM: Prețurile la raft vor co...,"Dictatorul Putin nu vrea pace, vrea capitulare...",other_external_actor,anti,acuzator,0,0,0,-2,0,Comentariul îl acuză pe Putin de intenții agre...,0.95,False
4,yt_G2org7gSZ0U_UgzIkCP3IgMROrwhTqJ4AaABAg,AlephNewsOfficial,,Ești de acord cu decizia lui Nicușor Dan de a-...,"Valoarea Președintelui ca om de stat, nu depin...",nicusor_dan,anti,acuzator,0,-1,0,0,0,Comentariul critică lipsa de competență și car...,0.95,False


In [38]:
# Tipologie rule-based pentru demo C4
# Intrare: target, stance, tone, institutional, legitimare, epistemic, geopolitic, mobilizare
# Ieșire: discourse_type, discourse_subtype, type_confidence
# Principiu: primul nod satisfăcut decide tipul. Ordinea regulilor contează.

SOVEREIGNTIST = {"georgescu", "simion", "sosoaca", "aur"}

INSTITUTIONAL = {
    "ccr", "alegeri", "guvern", "justitie", "parlament",
    "presedintie", "nicusor_dan", "bolojan", "other_state_institution"
}

PARTIES = {"psd", "pnl", "usr", "other_mainstream_actor"}

MEDIA = {
    "recorder", "g4media", "digi24",
    "presa_mainstream", "presa_investigativa", "other_media"
}

EXTERNAL = {"ue", "nato", "bruxelles", "other_external_actor"}


def assign_type(row):
    target = str(row.get("target", "none")).strip().lower()
    stance = str(row.get("stance", "none")).strip().lower()

    institutional = int(row.get("institutional", 0))
    legitimare = int(row.get("legitimare", 0))
    epistemic = int(row.get("epistemic", 0))
    geopolitic = int(row.get("geopolitic", 0))
    mobilizare = int(row.get("mobilizare", 0))

    activation = (
        abs(institutional)
        + abs(legitimare)
        + abs(epistemic)
        + abs(geopolitic)
        + abs(mobilizare)
    )

    is_sov = target in SOVEREIGNTIST
    is_inst = target in INSTITUTIONAL
    is_party = target in PARTIES
    is_media = target in MEDIA
    is_external = target in EXTERNAL

    # 1. T6 — fără țintă politică sau fără conținut discursiv articulat
    if target == "none" or stance == "none":
        return (
            "T6_afectiv_pozitional",
            "fara_tinta_politica_clara",
            "high"
        )

    if activation == 0:
        return (
            "T6_afectiv_pozitional",
            f"afectiv_{target}_{stance}",
            "high"
        )

    # 2. T1 — suport personalist
    # target suveranist + stance pro; personalismul dă claritatea tipului
    if is_sov and stance == "pro":
        if legitimare == -2:
            subtype = "personalism_cu_grievance" if institutional < 0 else "personalism_mesianic_pur"
            return ("T1_suport_personalist", subtype, "high")

        if legitimare == -1:
            return (
                "T1_suport_personalist",
                "suport_personalist_slab",
                "medium"
            )

        return (
            "T1_suport_personalist",
            "suport_afectiv_suveranist",
            "medium"
        )

    # 3. T2 — grievance anti-sistem
    # anti + institutional negativ + țintă instituțională / partid / media.
    # Dacă are și epistemic negativ, rămâne T2, dar subtip conspiratorial.
    if stance == "anti" and institutional < 0 and (is_inst or is_party or is_media):
        if epistemic < 0:
            return (
                "T2_grievance_anti_sistem",
                "grievance_conspiratorial",
                "high" if institutional == -2 else "medium"
            )

        if mobilizare > 0:
            return (
                "T2_grievance_anti_sistem",
                "grievance_mobilizator",
                "medium"
            )

        if is_media:
            return (
                "T2_grievance_anti_sistem",
                "grievance_anti_media",
                "medium"
            )

        return (
            "T2_grievance_anti_sistem",
            "grievance_pur",
            "high" if institutional == -2 else "medium"
        )

    # 4. T3 — opoziție suveranistă
    # anti-Georgescu/Simion/AUR/Șoșoacă.
    # Dacă are pol pozitiv pe procedură, dovezi, UE sau pluralism, e opoziție civică.
    if is_sov and stance == "anti":
        if institutional > 0 or legitimare > 0 or epistemic > 0 or geopolitic > 0:
            return (
                "T3_opozitie_suveranista",
                "opozitie_civic_procedurala",
                "high"
            )

        if institutional < 0:
            return (
                "T2_grievance_anti_sistem",
                "grievance_anti_suveranist",
                "medium"
            )

        return (
            "T3_opozitie_suveranista",
            "opozitie_difuza",
            "medium"
        )

    # 5. T4 — conspiraționism / externalism
    # Se aplică după T2/T3. Deci grievance instituțional conspiratorial a fost deja prins la T2.
    if epistemic < 0 or (geopolitic < 0 and stance == "anti"):
        if is_media and epistemic < 0:
            return (
                "T2_grievance_anti_sistem",
                "grievance_anti_media_conspiratorial",
                "medium"
            )

        if geopolitic == -2:
            return (
                "T4_conspiratie_externalism",
                "anti_externalism_geopolitic",
                "high"
            )

        if epistemic == -2 and geopolitic < 0:
            return (
                "T4_conspiratie_externalism",
                "conspiratie_externalista",
                "high"
            )

        if epistemic == -2:
            return (
                "T4_conspiratie_externalism",
                "conspiratie_difuza",
                "high"
            )

        return (
            "T4_conspiratie_externalism",
            "conspiratie_sau_externalism_slab",
            "medium"
        )

    # 6. T5 — pro-democratic / pro-european
    # pol pozitiv pe procedură, reguli, pluralism, verificare sau ancorare europeană.
    if institutional > 0 or legitimare > 0 or epistemic > 0 or geopolitic > 0:
        if geopolitic == 2:
            return (
                "T5_pro_democratic_european",
                "pro_european_ancorare",
                "high"
            )

        if institutional == 2:
            return (
                "T5_pro_democratic_european",
                "aparare_institutionala_procedurala",
                "high"
            )

        if legitimare > 0:
            return (
                "T5_pro_democratic_european",
                "legitimitate_pluralista",
                "medium"
            )

        if epistemic > 0:
            return (
                "T5_pro_democratic_european",
                "verificational_difuz",
                "low"
            )

        return (
            "T5_pro_democratic_european",
            "pro_democratic_difuz",
            "medium"
        )

    # 7. Fallback grievance
    if institutional < 0:
        return (
            "T2_grievance_anti_sistem",
            "grievance_rezidual",
            "low"
        )

    # 8. Fallback afectiv
    return (
        "T6_afectiv_pozitional",
        "afectiv_neclasificat",
        "low"
    )

In [39]:
df[["discourse_type", "discourse_subtype", "type_confidence"]] = df.apply(
    lambda row: pd.Series(assign_type(row)),
    axis=1
)

df["discourse_type"].value_counts()

discourse_type
T6_afectiv_pozitional         402
T2_grievance_anti_sistem        7
T4_conspiratie_externalism      4
T5_pro_democratic_european      3
T1_suport_personalist           3
T3_opozitie_suveranista         1
Name: count, dtype: int64

In [40]:
typed_path = Path("../data/typed/corpus_axis_typed.jsonl")
typed_path.parent.mkdir(parents=True, exist_ok=True)

df.to_json(
    typed_path,
    orient="records",
    lines=True,
    force_ascii=False
)

print("Saved:", typed_path)
print("Rows:", len(df))

Saved: ..\data\typed\corpus_axis_typed.jsonl
Rows: 420


## DBSCAN simplu pe cele 5 axe
Acum verificăm exploratoriu dacă valorile pe axe formează grupuri.
Important: DBSCAN nu produce tipologia teoretică. El grupează comentariile după proximitate numerică în spațiul axelor.
Tipologia rule-based rămâne metoda principală.

In [47]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
import pandas as pd

In [49]:
axis_cols = [
    "institutional",
    "legitimare",
    "epistemic",
    "geopolitic",
    "mobilizare"
]

X = df[axis_cols].fillna(0)
X_scaled = StandardScaler().fit_transform(X)

In [50]:
dbscan = DBSCAN(
    eps=0.9,
    min_samples=5
)

df["dbscan_cluster"] = dbscan.fit_predict(X_scaled)

df["dbscan_cluster"].value_counts().sort_index()

dbscan_cluster
-1     19
 0    401
Name: count, dtype: int64

In [51]:
# Cross-tab: tipologia rule-based vs DBSCAN
pd.crosstab(
    df["discourse_type"],
    df["dbscan_cluster"],
    margins=True
)

dbscan_cluster,-1,0,All
discourse_type,,,
T1_suport_personalist,3,0,3
T2_grievance_anti_sistem,7,0,7
T3_opozitie_suveranista,1,0,1
T4_conspiratie_externalism,4,0,4
T5_pro_democratic_european,3,0,3
T6_afectiv_pozitional,1,401,402
All,19,401,420


DBSCAN grupează comentariile doar după proximitatea numerică pe cele 5 axe: `institutional`, `legitimare`, `epistemic`, `geopolitic`, `mobilizare`. El nu știe teoria, nu știe ce înseamnă „grievance”, „personalism” sau „pro-democratic”. De aceea comparația cu `discourse_type` trebuie citită ca verificare exploratorie, nu ca validare finală.
Rezultatul arată o potrivire parțială:
- `cluster 0` este aproape perfect asociat cu `T6_afectiv_pozitional`. Asta sugerează că mesajele afective sau slab articulate formează un grup numeric clar: au activare redusă pe axe.
- Unele clustere mici captează fragmente din `T1_suport_personalist` și `T2_grievance_anti_sistem`. Asta arată că personalismul și grievance-ul apar în forme relativ coerente, dar nu într-un singur bloc compact.
- `T5_pro_democratic_european` apare aproape complet în `-1`, adică în zona de noise. Asta nu înseamnă că tipul este greșit, ci că este rar, dispersat sau exprimat în forme variate.
`-1` în DBSCAN înseamnă „noise”: observații care nu aparțin unei regiuni dense. În tabel, 85 din 180 comentarii sunt în `-1`, deci aproape jumătate din corpusul testat nu formează clustere dense pe cele 5 axe.
Concluzie: tipologia rule-based și DBSCAN răspund la întrebări diferite. Tipologia construiește clase interpretative pe baza teoriei și a regulilor explicite. DBSCAN caută densități numerice în date. Dacă se suprapun, avem o confirmare exploratorie. Dacă nu se suprapun, nu înseamnă automat că una este greșită; poate însemna că un tip discursiv este rar, mixt sau teoretic important, dar numeric dispersat.
Pentru proiect, păstrăm tipologia rule-based ca metodă principală și folosim clustering-ul doar ca instrument exploratoriu.

In [52]:
# Profil mediu pe fiecare cluster
cluster_profile = (
    df.groupby("dbscan_cluster")[axis_cols]
      .mean()
      .round(2)
)

cluster_profile

,institutional,legitimare,epistemic,geopolitic,mobilizare
dbscan_cluster,,,,,
-1,-0.37,-0.05,-0.68,-0.16,0.26
0,0.00,0.00,0.00,0.00,0.00


## Cum citim profilul clusterelor DBSCAN
DBSCAN nu construiește tipologia teoretică, ci grupează comentariile după valori similare pe cele 5 axe.
- `cluster 0`: toate axele = 0 → comentarii afective / slab articulate (`T6`)
- `cluster 1`: `geopolitic = -1` → anti-UE/NATO/Bruxelles slab
- `cluster 2`: `legitimare = -1` → personalism slab
- `cluster 3`: `institutional = -1` → grievance instituțional slab
- `cluster 4`: `legitimare = -2` → personalism puternic
- `cluster 5`: `institutional = -2`, `epistemic = -1` → grievance puternic + suspiciune
- `cluster 6`: `epistemic = -1`, `geopolitic = -1` → suspiciune + anti-externalism slab
- `cluster 7`: `epistemic = -1` → suspiciune / manipulare difuză
- `cluster -1`: noise → cazuri mixte, dispersate, fără grup dens clar
Concluzie: DBSCAN prinde bine profilurile simple pe axe, dar pune multe cazuri mixte la `-1`. De aceea îl folosim exploratoriu, nu ca înlocuitor pentru tipologia rule-based.

In [53]:
# Exemple din fiecare cluster
for cluster_id in sorted(df["dbscan_cluster"].unique()):
    print("\n" + "=" * 80)
    print("DBSCAN cluster:", cluster_id)
    print("=" * 80)

    sample = df[df["dbscan_cluster"] == cluster_id].sample(
        min(3, len(df[df["dbscan_cluster"] == cluster_id])),
        random_state=42
    )

    for _, row in sample.iterrows():
        print()
        print("Tip rule-based:", row.get("discourse_type"))
        print("Axes:", {col: row[col] for col in axis_cols})
        print(row["text"][:300])


DBSCAN cluster: -1

Tip rule-based: T3_opozitie_suveranista
Axes: {'institutional': 0, 'legitimare': 0, 'epistemic': -1, 'geopolitic': 0, 'mobilizare': 0}
jigodia aia de Georgescu nu lua intrebari inca din campanie, cand NU AVEA PUTEREA ,cum zic americanii writings were on the wall'' scria clar pe perete ce va face avand puterea

Tip rule-based: T2_grievance_anti_sistem
Axes: {'institutional': -1, 'legitimare': 0, 'epistemic': -1, 'geopolitic': 0, 'mobilizare': 0}
Iar din nr acela 80 % dintre ei sau uitat la nea Ilie sa vadă ce dezastre mai aduce pe capul nostru! Acele doua personaje ilegitime, așa cum ne dovedesc prin ce a ce fac ei ,ne vor dispariția noastră ca țară și popor. Nu ai cum sa înțelegi comportamentul lor. Nu au planuri de a deschide locuri de mu

Tip rule-based: T1_suport_personalist
Axes: {'institutional': 0, 'legitimare': -1, 'epistemic': 0, 'geopolitic': 0, 'mobilizare': 2}
POPORUL ROMÂN VĂ AȘTEAPTĂ, DOMNULE PREȘEDINTE,,,,SĂ FACEM ROMÂNIA MARE,,,,

DBSCAN cluster: 0

